This notebook covers importing the dataset from https://www.kaggle.com/datasets/patrickfleith/nasa-battery-dataset which was adapted/streamlined from B. Saha and K. Goebel (2007). “Battery Data Set”, NASA Prognostics Data Repository, NASA Ames Research Center, Moffett Field, CA. 

# 1. Setup & Imports

In [2]:
import os
import re
from datetime import datetime

import pandas as pd
import kagglehub


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Load & Parse Raw NASA Battery Data
A user named Patrick Fleith had kindly converted the original MATLAB files into CSV. This section not only converts MATLAB datenum vectors to Python `datetime`, it also then loop through every file/cycle to build three raw DataFrames: `charge_df`, `discharge_df`, `eis_df`.

In [3]:
DATA_DIR = kagglehub.dataset_download("patrickfleith/nasa-battery-dataset")
print(os.listdir(DATA_DIR))

['cleaned_dataset']


In [4]:
root = os.path.join(DATA_DIR, "cleaned_dataset")
print(os.listdir(root))

metadata = pd.read_csv(os.path.join(root, "metadata.csv"))
print(metadata.shape)
print(metadata.columns.tolist())
print(metadata.head())

# peek at one cycle file to see its columns
data_dir = os.path.join(root, "data")
sample_file = os.listdir(data_dir)[0]
sample = pd.read_csv(os.path.join(data_dir, sample_file))
print(sample_file)
print(sample.columns.tolist())
print(sample.head())

['data', 'extra_infos', 'metadata.csv']
(7565, 10)
['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct']
        type                                         start_time  \
0  discharge  [2010.       7.      21.      15.       0.    ...   
1  impedance  [2010.       7.      21.      16.      53.    ...   
2     charge  [2010.       7.      21.      17.      25.    ...   
3  impedance                    [2010    7   21   20   31    5]   
4  discharge  [2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...   

   ambient_temperature battery_id  test_id  uid   filename  \
0                    4      B0047        0    1  00001.csv   
1                   24      B0047        1    2  00002.csv   
2                    4      B0047        2    3  00003.csv   
3                   24      B0047        3    4  00004.csv   
4                    4      B0047        4    5  00005.csv   

             Capacity                   Re                

In [5]:
#creating dataframes for each state (charging, discharging, and EIS measurement)
metadata["Capacity"] = pd.to_numeric(metadata["Capacity"], errors="coerce")
metadata["Re"] = pd.to_numeric(metadata["Re"], errors="coerce")
metadata["Rct"] = pd.to_numeric(metadata["Rct"], errors="coerce")


def parse_start_time(s):
    """Parse the dataset's MATLAB-style date-vector string into a datetime."""
    if not isinstance(s, str) or not s.strip():
        return None
    inner = s.strip().strip("[]")
    inner = re.sub(r"[,\s]+", ",", inner.strip())
    parts = [float(x) for x in inner.split(",") if x]
    if len(parts) < 6:
        return None
    yr, mo, dy, hr, mi, sc = parts[:6]
    return datetime(int(yr), int(mo), int(dy), int(hr), int(mi), int(sc))


charge_frames, discharge_frames, eis_frames = [], [], []

for row in metadata.itertuples():
    cycle_path = os.path.join(data_dir, row.filename)
    df = pd.read_csv(cycle_path)
    n = len(df)

    cycle_start = parse_start_time(row.start_time)
    df["Cell_ID"] = row.battery_id
    df["Cycle_Number"] = row.test_id
    df["Ambient_Temperature"] = row.ambient_temperature
    df["Time_abs"] = cycle_start

    if row.type == "charge":
        charge_frames.append(df)
    elif row.type == "discharge":
        df["Capacity"] = row.Capacity
        discharge_frames.append(df)
    elif row.type == "impedance":
        for col in ["Sense_current", "Battery_current", "Current_ratio",
                    "Battery_impedance", "Rectified_Impedance"]:
            df[col] = df[col].apply(complex)
        df["Re"] = row.Re
        df["Rct"] = row.Rct
        eis_frames.append(df)

charge_df = pd.concat(charge_frames, ignore_index=True)
discharge_df = pd.concat(discharge_frames, ignore_index=True)
eis_df = pd.concat(eis_frames, ignore_index=True)

print(charge_df.shape, discharge_df.shape, eis_df.shape)

(6512876, 10) (770070, 11) (93888, 11)


In [6]:
#further peeking into the metadata to see the headers and values used in the data
extra_dir = os.path.join(root, "extra_infos")
print(os.listdir(extra_dir))

imp_row = metadata[metadata['type'] == 'impedance'].iloc[0]
imp_sample = pd.read_csv(os.path.join(root, "data", imp_row['filename']))
print(imp_row['filename'])
print(imp_sample.columns.tolist())
print(imp_sample.head())

['README_05_06_07_18.txt', 'README_25_26_27_28.txt', 'README_29_30_31_32.txt', 'README_33_34_36.txt', 'README_38_39_40.txt', 'README_41_42_43_44.txt', 'README_45_46_47_48.txt', 'README_49_50_51_52.txt', 'README_53_54_55_56.txt']
00002.csv
['Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance']
                             Sense_current  \
0  (928.3472290039062-48.499576568603516j)   
1   (922.0059814453125-52.15481185913086j)   
2   (920.0405883789062-52.19926452636719j)   
3  (914.2005615234375-52.371726989746094j)   
4  (911.2781372070312-45.477928161621094j)   

                           Battery_current  \
0      (228.7861328125-70.94481658935547j)   
1  (233.29783630371094-67.84492492675781j)   
2  (236.62075805664062-67.22105407714844j)   
3   (242.2973175048828-66.45936584472656j)   
4   (246.62347412109375-65.4150161743164j)   

                              Current_ratio  \
0    (3.76172584438583+0.9544956717274271j)   
1    (3.7038357

In [7]:
#prints all the readme (experimental notes for each battery dataset)
for f in sorted(os.listdir(extra_dir)):
    print(f"===== {f} =====")
    print(open(os.path.join(extra_dir, f), errors='ignore').read())
    print()

===== README_05_06_07_18.txt =====
Data Description:
A set of four Li-ion batteries (# 5, 6, 7 and 18) were run through 3 different operational profiles (charge, discharge and impedance) at room temperature. Charging was carried out in a constant current (CC) mode at 1.5A until the battery voltage reached 4.2V and then continued in a constant voltage (CV) mode until the charge current dropped to 20mA. Discharge was carried out at a constant current (CC) level of 2A until the battery voltage fell to 2.7V, 2.5V, 2.2V and 2.5V for batteries 5 6 7 and 18 respectively. Impedance measurement was carried out through an electrochemical impedance spectroscopy (EIS) frequency sweep from 0.1Hz to 5kHz. Repeated charge and discharge cycles result in accelerated aging of the batteries while impedance measurements provide insight into the internal battery parameters that change as aging progresses. The experiments were stopped when the batteries reached end-of-life (EOL) criteria, which was a 30% fa

Due to control software crashing (as noted in the readme file), batteries 49-52 will not be included in the dataset.

In [8]:
# Drop cells that failed early due to control-software crashes (B0049-B0052)
dropped_cells = ['B0049', 'B0050', 'B0051', 'B0052']
eis_df = eis_df[~eis_df['Cell_ID'].isin(dropped_cells)]
charge_df = charge_df[~charge_df['Cell_ID'].isin(dropped_cells)]
discharge_df = discharge_df[~discharge_df['Cell_ID'].isin(dropped_cells)]

In [9]:
#putting battery experimental conditions into a nested dictionary before creating a dataframe
battery_protocol = {
    # Batteries 5,6,7,18 — CC discharge 2A, room temp
    'B0005': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 24, 'cutoff_V': 2.7, 'endoflife_criteria': 0.30},
    'B0006': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 24, 'cutoff_V': 2.5, 'endoflife_criteria': 0.30},
    'B0007': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 24, 'cutoff_V': 2.2, 'endoflife_criteria': 0.30},
    'B0018': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 24, 'cutoff_V': 2.5, 'endoflife_criteria': 0.30},

    # Batteries 25-28 — square wave discharge, 4A amplitude, 50% duty, room temp
    'B0025': {'discharge_mode': 'square_wave', 'discharge_current_A': 4, 'ambient_C': 24, 'cutoff_V': 2.0, 'endoflife_criteria': 0.30},
    'B0026': {'discharge_mode': 'square_wave', 'discharge_current_A': 4, 'ambient_C': 24, 'cutoff_V': 2.2, 'endoflife_criteria': 0.30},
    'B0027': {'discharge_mode': 'square_wave', 'discharge_current_A': 4, 'ambient_C': 24, 'cutoff_V': 2.5, 'endoflife_criteria': 0.30},
    'B0028': {'discharge_mode': 'square_wave', 'discharge_current_A': 4, 'ambient_C': 24, 'cutoff_V': 2.7, 'endoflife_criteria': 0.30},

    # Batteries 29-32 — CC discharge 4A, elevated temp (43C)
    'B0029': {'discharge_mode': 'CC', 'discharge_current_A': 4, 'ambient_C': 43, 'cutoff_V': 2.0, 'endoflife_criteria': 0.30},
    'B0030': {'discharge_mode': 'CC', 'discharge_current_A': 4, 'ambient_C': 43, 'cutoff_V': 2.2, 'endoflife_criteria': 0.30},
    'B0031': {'discharge_mode': 'CC', 'discharge_current_A': 4, 'ambient_C': 43, 'cutoff_V': 2.5, 'endoflife_criteria': 0.30},
    'B0032': {'discharge_mode': 'CC', 'discharge_current_A': 4, 'ambient_C': 43, 'cutoff_V': 2.7, 'endoflife_criteria': 0.30},

    # Batteries 33,34,36 — CC discharge, room temp, EOL criteria: 20%
    'B0033': {'discharge_mode': 'CC', 'discharge_current_A': 4, 'ambient_C': 24, 'cutoff_V': 2.0, 'endoflife_criteria': 0.20},
    'B0034': {'discharge_mode': 'CC', 'discharge_current_A': 4, 'ambient_C': 24, 'cutoff_V': 2.2, 'endoflife_criteria': 0.20},
    'B0036': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 24, 'cutoff_V': 2.7, 'endoflife_criteria': 0.20},

    # Batteries 38-40 — multiple load levels (1,2,4A), mixed temps, EOL criteria: 20%
    'B0038': {'discharge_mode': 'multi_level', 'discharge_current_A': None, 'ambient_C': None, 'cutoff_V': 2.2, 'endoflife_criteria': 0.20},
    'B0039': {'discharge_mode': 'multi_level', 'discharge_current_A': None, 'ambient_C': None, 'cutoff_V': 2.5, 'endoflife_criteria': 0.20},
    'B0040': {'discharge_mode': 'multi_level', 'discharge_current_A': None, 'ambient_C': None, 'cutoff_V': 2.7, 'endoflife_criteria': 0.20},

    # Batteries 41-44 — fixed load levels (4A and 1A), 4C ambient, EOL criteria: 30%
    'B0041': {'discharge_mode': 'fixed_load', 'discharge_current_A': None, 'ambient_C': 4, 'cutoff_V': 2.0, 'endoflife_criteria': 0.30},
    'B0042': {'discharge_mode': 'fixed_load', 'discharge_current_A': None, 'ambient_C': 4, 'cutoff_V': 2.2, 'endoflife_criteria': 0.30},
    'B0043': {'discharge_mode': 'fixed_load', 'discharge_current_A': None, 'ambient_C': 4, 'cutoff_V': 2.5, 'endoflife_criteria': 0.30},
    'B0044': {'discharge_mode': 'fixed_load', 'discharge_current_A': None, 'ambient_C': 4, 'cutoff_V': 2.7, 'endoflife_criteria': 0.30},

    # Batteries 45-48 — fixed load 1A, 4C ambient, EOL criteria: 30%
    'B0045': {'discharge_mode': 'CC', 'discharge_current_A': 1, 'ambient_C': 4, 'cutoff_V': 2.0, 'endoflife_criteria': 0.30},
    'B0046': {'discharge_mode': 'CC', 'discharge_current_A': 1, 'ambient_C': 4, 'cutoff_V': 2.2, 'endoflife_criteria': 0.30},
    'B0047': {'discharge_mode': 'CC', 'discharge_current_A': 1, 'ambient_C': 4, 'cutoff_V': 2.5, 'endoflife_criteria': 0.30},
    'B0048': {'discharge_mode': 'CC', 'discharge_current_A': 1, 'ambient_C': 4, 'cutoff_V': 2.7, 'endoflife_criteria': 0.30},

    # Batteries 53-56 — fixed load 2A, 4C ambient, EOL criteria: 30%
    'B0053': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 4, 'cutoff_V': 2.0, 'endoflife_criteria': 0.30},
    'B0054': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 4, 'cutoff_V': 2.2, 'endoflife_criteria': 0.30},
    'B0055': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 4, 'cutoff_V': 2.5, 'endoflife_criteria': 0.30},
    'B0056': {'discharge_mode': 'CC', 'discharge_current_A': 2, 'ambient_C': 4, 'cutoff_V': 2.7, 'endoflife_criteria': 0.30},
}

protocol_df = pd.DataFrame.from_dict(battery_protocol, orient='index').reset_index()
protocol_df = protocol_df.rename(columns={'index': 'Cell_ID'})

discharge_df = discharge_df.merge(protocol_df, on='Cell_ID', how='left')

In [10]:
# Sanity check: confirm dtypes and non-null counts after building the raw dataframes
print("Charge Dataframe:")
print(charge_df.info())
print("\nDischarge Dataframe:")
print(discharge_df.info())
print("\nEIS Dataframe:")
print(eis_df.info())

Charge Dataframe:
<class 'pandas.DataFrame'>
Index: 6206968 entries, 0 to 6512875
Data columns (total 10 columns):
 #   Column                Dtype         
---  ------                -----         
 0   Voltage_measured      float64       
 1   Current_measured      float64       
 2   Temperature_measured  float64       
 3   Current_charge        float64       
 4   Voltage_charge        float64       
 5   Time                  float64       
 6   Cell_ID               str           
 7   Cycle_Number          int64         
 8   Ambient_Temperature   int64         
 9   Time_abs              datetime64[us]
dtypes: datetime64[us](1), float64(6), int64(2), str(1)
memory usage: 520.9 MB
None

Discharge Dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 746334 entries, 0 to 746333
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Voltage_measured      746334 non-null  float64      

# 3. Installing pyarrow to allow saving dataframes as pickles

In [14]:
%pip install "pyarrow<21"

   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   - -------------------------------------- 1.0/25.7 MB 12.5 MB/s eta 0:00:02
   -- ------------------------------------- 1.6/25.7 MB 6.0 MB/s eta 0:00:05
   --- ------------------------------------ 2.1/25.7 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.1/25.7 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.1/25.7 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.1/25.7 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.1/25.7 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/25.7 MB 1.4 MB/s eta 0:00:17
   ----- ---------------------------------- 3.7/25.7 MB 2.0 MB/s eta 0:00:11
   ------ --------------------------------- 4.2/25.7 MB 2.0 MB/s eta 0:00:11
   ------ --------------------------------- 4.5/25.7 MB 2.0 MB/s eta 0:00:11
   ------- -------------------------------- 5.0/25.7 MB 2.0 MB/s eta 0:00:11
   --

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
charge_df.to_pickle("charge_df.pkl")
discharge_df.to_pickle("discharge_df.pkl")
eis_df.to_pickle("eis_df.pkl")

The exploratory data analysis will be covered in the next notebook file.